In [5]:
%matplotlib notebook

In [6]:
# ==============================================================================
# ⚙️ USER CONFIGURATION SECTION
# ==============================================================================
RAW_INPUT_DIR = r"C:\Users\DELL\Documents\GitHub\fyp\05_Data_Storage\02_Verified_Raw"
BASE_OUTPUT_DIR = r"C:\Users\DELL\Documents\GitHub\fyp\05_Data_Storage\03_Windowed"

WINDOW_DURATION = 15     # Seconds (blue window width)
FS              = 400    # Hz
VIEW_WINDOW_SEC = 25     # Seconds shown in sliding viewer
# ==============================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# ✅ FORCE A REAL GUI WINDOW BACKEND (TkAgg)
# ------------------------------------------------------------------
import matplotlib
matplotlib.use("TkAgg")  # MUST be before importing pyplot

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from tkinter import filedialog, Tk


# ------------------------------------------------------------------------------
# FILE LOADING / SELECTION
# ------------------------------------------------------------------------------
def load_csv_with_aliases(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]

    col_map = {c.strip().lower(): c.strip() for c in df.columns}
    rename_dict = {}

    # IR aliases
    if "ir_value" in col_map:
        rename_dict[col_map["ir_value"]] = "IR_Value"
    elif "ir" in col_map:
        rename_dict[col_map["ir"]] = "IR_Value"
    elif "infrared" in col_map:
        rename_dict[col_map["infrared"]] = "IR_Value"

    # RED aliases
    if "red_value" in col_map:
        rename_dict[col_map["red_value"]] = "Red_Value"
    elif "red" in col_map:
        rename_dict[col_map["red"]] = "Red_Value"

    if rename_dict:
        df = df.rename(columns=rename_dict)

    required_cols = ["IR_Value", "Red_Value"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Required columns missing: {missing}\n"
            f"Available columns: {list(df.columns)}\n"
            f"Expected: Timestamp, IR, RED (or IR_Value, Red_Value)"
        )

    return df


def choose_file(raw_input_dir: str) -> str:
    if os.path.isfile(raw_input_dir) and raw_input_dir.lower().endswith(".csv"):
        print(f"📄 Direct file detected: {os.path.basename(raw_input_dir)}")
        return raw_input_dir

    if os.path.isdir(raw_input_dir):
        print("📂 Opening File Dialog...")
        root = Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        file_path = filedialog.askopenfilename(
            initialdir=raw_input_dir,
            title="Select Raw CSV",
            filetypes=[("CSV files", "*.csv")]
        )

        root.destroy()
        return file_path

    print(f"❌ Error: Path '{raw_input_dir}' not found.")
    return ""


# ------------------------------------------------------------------------------
# FIXED WINDOW SELECTOR (CLEAN VIEW: RED TOP, IR BOTTOM)
# ------------------------------------------------------------------------------
def fixed_window_selector(df: pd.DataFrame, total_duration: float):
    """
    Clean layout:
      - RED on TOP subplot (red line)
      - IR on BOTTOM subplot (blue line)
      - shared X-axis (time)
      - light-blue window span shown on BOTH subplots
      - click on either subplot to move window start
      - Add Block stores (start,end) based on window span
    """
    view_sec = float(VIEW_WINDOW_SEC)
    win_sec = float(WINDOW_DURATION)

    if view_sec <= 2:
        view_sec = 10.0
    if win_sec <= 0:
        raise ValueError("WINDOW_DURATION must be > 0")

    max_scroll_start = max(0.0, total_duration - view_sec)
    saved_windows = []

    def clamp_window_start(s: float) -> float:
        return float(np.clip(s, 0.0, max(0.0, total_duration - win_sec)))

    def sec_to_idx(t_sec: float) -> int:
        return int(np.clip(t_sec * FS, 0, len(df)))

    def get_segment(t0: float, t1: float):
        s = sec_to_idx(t0)
        e = sec_to_idx(t1)
        seg = df.iloc[s:e].copy()
        tt = np.linspace(t0, t1, max(1, len(seg)))
        return tt, seg

    # ---- Figure with 2 stacked axes (shared x) ----
    fig = plt.figure(figsize=(13, 7))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1, 1], hspace=0.08)

    ax_red = fig.add_subplot(gs[0, 0])  # TOP
    ax_ir  = fig.add_subplot(gs[1, 0], sharex=ax_red)  # BOTTOM

    fig.suptitle("Fixed Window Selector: click to place blue window, then 'Add Block'")

    ax_red.set_ylabel("RED Amplitude")
    ax_ir.set_ylabel("IR Amplitude")
    ax_ir.set_xlabel("Time (seconds)")

    ax_red.grid(True)
    ax_ir.grid(True)

    # Initial view
    scroll_start = 0.0
    scroll_end = min(total_duration, scroll_start + view_sec)

    t, seg = get_segment(scroll_start, scroll_end)

    # ✅ Colors: RED in red, IR in blue
    (line_red,) = ax_red.plot(t, seg["Red_Value"].values, color="red",  label="RED", alpha=0.85)
    (line_ir,)  = ax_ir.plot(t,  seg["IR_Value"].values,  color="blue", label="IR",  alpha=0.85)

    ax_red.legend(loc="upper right")
    ax_ir.legend(loc="upper right")

    # Blue window (span) on both axes
    window_start = clamp_window_start(scroll_start)
    window_end = window_start + win_sec

    blue_patch_red = ax_red.axvspan(window_start, window_end, alpha=0.18)
    blue_patch_ir  = ax_ir.axvspan(window_start, window_end, alpha=0.18)

    # Info box on bottom axis (less clutter)
    info_text = ax_ir.text(
        0.01, 0.03, "",
        transform=ax_ir.transAxes, fontsize=10,
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="none")
    )

    def autoscale_axis(ax, y):
        if len(y) == 0:
            return
        y_min = float(np.nanmin(y))
        y_max = float(np.nanmax(y))
        if np.isfinite(y_min) and np.isfinite(y_max):
            if y_max > y_min:
                pad = 0.07 * (y_max - y_min)
                ax.set_ylim(y_min - pad, y_max + pad)
            else:
                ax.set_ylim(y_min - 1, y_max + 1)

    def update_info():
        we = window_start + win_sec
        info_text.set_text(f"Blue Window: {window_start:.3f}s → {we:.3f}s | Saved: {len(saved_windows)}")
        fig.canvas.draw_idle()

    def redraw_window_spans():
        nonlocal blue_patch_red, blue_patch_ir
        we = window_start + win_sec

        try:
            blue_patch_red.remove()
        except Exception:
            pass
        try:
            blue_patch_ir.remove()
        except Exception:
            pass

        blue_patch_red = ax_red.axvspan(window_start, we, alpha=0.18)
        blue_patch_ir  = ax_ir.axvspan(window_start, we, alpha=0.18)
        update_info()

    # Slider
    ax_slider = fig.add_axes([0.10, 0.12, 0.80, 0.035])
    slider = Slider(
        ax=ax_slider,
        label="Scroll Start (sec)",
        valmin=0.0,
        valmax=max_scroll_start if max_scroll_start > 0 else 0.0,
        valinit=0.0,
        valstep=max(1.0 / FS, 0.01)
    )

    def on_slider_change(val):
        nonlocal scroll_start, scroll_end, window_start
        scroll_start = float(val)
        scroll_end = min(total_duration, scroll_start + view_sec)

        tt, seg2 = get_segment(scroll_start, scroll_end)

        line_red.set_data(tt, seg2["Red_Value"].values)
        line_ir.set_data(tt,  seg2["IR_Value"].values)

        ax_ir.set_xlim(tt[0], tt[-1] if len(tt) > 1 else scroll_end)

        autoscale_axis(ax_red, seg2["Red_Value"].values)
        autoscale_axis(ax_ir,  seg2["IR_Value"].values)

        # If window is fully outside view, snap it to the left of the visible region
        if (window_start + win_sec) < scroll_start or window_start > scroll_end:
            window_start = clamp_window_start(scroll_start)

        redraw_window_spans()
        fig.canvas.draw_idle()

    slider.on_changed(on_slider_change)

    # Click on either subplot to place the window start
    def on_click(event):
        nonlocal window_start
        if event.inaxes not in (ax_red, ax_ir):
            return
        if event.xdata is None:
            return
        window_start = clamp_window_start(float(event.xdata))
        redraw_window_spans()

    fig.canvas.mpl_connect("button_press_event", on_click)

    # Buttons
    ax_add  = fig.add_axes([0.10, 0.03, 0.18, 0.06])
    ax_undo = fig.add_axes([0.31, 0.03, 0.18, 0.06])
    ax_done = fig.add_axes([0.74, 0.03, 0.20, 0.06])

    btn_add  = Button(ax_add, "Add Block")
    btn_undo = Button(ax_undo, "Undo Last")
    btn_done = Button(ax_done, "Done")

    def add_block(_):
        nonlocal window_start
        s = float(window_start)
        e = float(window_start + win_sec)

        if e > total_duration:
            print("⚠️ Window exceeded signal end. Adjusting.")
            s = clamp_window_start(total_duration - win_sec)
            e = s + win_sec
            window_start = s
            redraw_window_spans()

        saved_windows.append((s, e))
        print(f"✅ Saved Window: {s:.3f}s - {e:.3f}s")
        update_info()

    def undo_last(_):
        if not saved_windows:
            print("❌ No saved windows to undo.")
            return
        removed = saved_windows.pop()
        print(f"↩️ Removed: {removed[0]:.3f}s - {removed[1]:.3f}s")
        update_info()

    def done(_):
        plt.close(fig)

    btn_add.on_clicked(add_block)
    btn_undo.on_clicked(undo_last)
    btn_done.on_clicked(done)

    # Initial autoscale and info
    autoscale_axis(ax_red, seg["Red_Value"].values)
    autoscale_axis(ax_ir,  seg["IR_Value"].values)
    update_info()

    plt.show(block=True)
    return saved_windows


# ------------------------------------------------------------------------------
# SAVE THE WINDOWS
# ------------------------------------------------------------------------------
def save_selected_windows(df: pd.DataFrame, file_path: str, windows: list):
    file_name_clean = os.path.basename(file_path).replace(".csv", "")
    specific_output_folder = os.path.join(BASE_OUTPUT_DIR, file_name_clean)
    os.makedirs(specific_output_folder, exist_ok=True)

    print(f"\n💾 Saving {len(windows)} selected windows (each = {WINDOW_DURATION}s)...")

    for idx, (start_sec, end_sec) in enumerate(windows):
        start_idx = int(start_sec * FS)
        end_idx   = int(end_sec * FS)

        chunk = df.iloc[start_idx:end_idx].copy()
        out_name = f"{file_name_clean}_Win{idx}.csv"
        out_path = os.path.join(specific_output_folder, out_name)
        chunk.to_csv(out_path, index=False)

        print(f"   [{idx}] Saved: {start_sec:.3f}s - {end_sec:.3f}s -> {out_name}")

    print("\n" + "=" * 60)
    print("✅ DONE!")
    print(f"📂 Location: {specific_output_folder}")
    print("=" * 60)


# ------------------------------------------------------------------------------
# MAIN
# ------------------------------------------------------------------------------
def slice_data():
    print("✅ Matplotlib backend:", matplotlib.get_backend())

    file_path = choose_file(RAW_INPUT_DIR)
    if not file_path:
        print("❌ No file selected.")
        return

    try:
        df = load_csv_with_aliases(file_path)
        total_duration = len(df) / FS
        print(f"✅ Loaded: {os.path.basename(file_path)} | Duration: {total_duration:.2f}s | FS={FS} Hz")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return

    print("\n🧭 Controls:")
    print("   - Use SLIDER to scroll")
    print("   - CLICK on RED(top) or IR(bottom) plot to place BLUE window start")
    print("   - 'Add Block' saves EXACTLY the blue-window segment")
    print("   - 'Undo Last' removes last saved window")
    print("   - 'Done' closes viewer and saves to files\n")

    windows = fixed_window_selector(df, total_duration)

    if not windows:
        print("❌ No windows selected. Exiting.")
        return

    print("\n✅ Selected Windows:")
    for k, (s, e) in enumerate(windows):
        print(f"   {k+1}. {s:.3f}s - {e:.3f}s")

    save_selected_windows(df, file_path, windows)


if __name__ == "__main__":
    slice_data()

✅ Matplotlib backend: TkAgg
📂 Opening File Dialog...


✅ Loaded: abeeha(23-enc-19)v5.csv | Duration: 114.70s | FS=400 Hz

🧭 Controls:
   - Use SLIDER to scroll
   - CLICK on RED(top) or IR(bottom) plot to place BLUE window start
   - 'Add Block' saves EXACTLY the blue-window segment
   - 'Undo Last' removes last saved window
   - 'Done' closes viewer and saves to files

✅ Saved Window: 5.842s - 20.842s
✅ Saved Window: 18.473s - 33.473s
✅ Saved Window: 23.139s - 38.139s
✅ Saved Window: 26.373s - 41.373s
✅ Saved Window: 30.021s - 45.021s
✅ Saved Window: 34.165s - 49.165s
✅ Saved Window: 40.071s - 55.071s
✅ Saved Window: 45.050s - 60.050s
✅ Saved Window: 50.062s - 65.062s
✅ Saved Window: 54.999s - 69.999s
✅ Saved Window: 60.061s - 75.061s
✅ Saved Window: 65.043s - 80.043s
✅ Saved Window: 70.055s - 85.055s
✅ Saved Window: 74.916s - 89.916s
✅ Saved Window: 80.052s - 95.052s
✅ Saved Window: 85.015s - 100.015s
✅ Saved Window: 90.006s - 105.006s
✅ Saved Window: 94.994s - 109.994s
✅ Saved Window: 99.698s - 114.698s
↩️ Removed: 99.698s - 114.698s
↩️